In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [3]:
df = pd.read_csv("../data/heart_disease.csv")

print(df.shape)
df.head()

(920, 16)


,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [4]:
# Invalid clinical values → missing
df.loc[df["chol"] <= 0, "chol"] = np.nan
df.loc[df["trestbps"] <= 0, "trestbps"] = np.nan

# Convert original multi-class target into binary target
df["target"] = (df["num"] > 0).astype(int)

# Remove identifier
df = df.drop(columns=["id"])

In [5]:
X = df.drop(columns=["num", "target", "dataset"])
y = df["target"]

In [6]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print(X.columns.tolist())

X shape: (920, 13)
y shape: (920,)
['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalch', 'exang', 'oldpeak', 'slope', 'ca', 'thal']


In [7]:
numerical_features = [
    "age",
    "trestbps",
    "chol",
    "thalch",
    "oldpeak"
]

categorical_features = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal"
]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [9]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nTrain target:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTest target:")
print(y_test.value_counts(normalize=True).round(3))

X_train: (736, 13)
X_test : (184, 13)

Train target:
target
1    0.553
0    0.447
Name: proportion, dtype: float64

Test target:
target
1    0.554
0    0.446
Name: proportion, dtype: float64


In [10]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [11]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        drop="first"
    ))
])

In [12]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [13]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [14]:
print("Before preprocessing:")
print("Train:", X_train.shape)
print("Test :", X_test.shape)

print("\nAfter preprocessing:")
print("Train:", X_train_processed.shape)
print("Test :", X_test_processed.shape)

Before preprocessing:
Train: (736, 13)
Test : (184, 13)

After preprocessing:
Train: (736, 20)
Test : (184, 20)


In [15]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))
print(feature_names)

Number of processed features: 20
['num__age' 'num__trestbps' 'num__chol' 'num__thalch' 'num__oldpeak'
 'cat__sex_Male' 'cat__cp_atypical angina' 'cat__cp_non-anginal'
 'cat__cp_typical angina' 'cat__fbs_True' 'cat__restecg_normal'
 'cat__restecg_st-t abnormality' 'cat__exang_True' 'cat__slope_flat'
 'cat__slope_upsloping' 'cat__ca_1.0' 'cat__ca_2.0' 'cat__ca_3.0'
 'cat__thal_normal' 'cat__thal_reversable defect']


In [18]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("scaler", StandardScaler())
])

In [19]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Missing"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        drop="first"
    ))
])

In [20]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [ ]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Missing"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        drop="first"
    ))
])